# AFEC-X 2026
## Unsupervised Learning with Flower Images

In the previous activity with Teachable Machine, **we defined the classes first**.

Now we will give the computer flower images **without species labels** and ask it to find groups.

### Workflow

**Images → Numerical features → Feature space → K-means clustering → Human interpretation**

### Main question

> If the machine finds groups, do those groups automatically represent biological species?


## Step 1 — Import packages

No GPU, PyTorch, TensorFlow, or CUDA is required.


In [ ]:
from pathlib import Path
import math

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image, ImageOps

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans


## Step 2 — Locate the dataset

The flower images are stored in:

```text
data/unsupervised/flowers/
```

The filenames do not contain species names.


In [ ]:
HERE = Path.cwd()

if (HERE / "data").exists():
    ROOT = HERE
elif (HERE.parent / "data").exists():
    ROOT = HERE.parent
else:
    raise FileNotFoundError("Cannot find the repository 'data' folder.")

IMAGE_DIR = ROOT / "data" / "unsupervised" / "flowers"

extensions = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}
image_files = sorted(
    p for p in IMAGE_DIR.rglob("*")
    if p.is_file() and p.suffix.lower() in extensions
)

print("Repository root:", ROOT.resolve())
print("Flower images:", len(image_files))

if len(image_files) < 4:
    raise FileNotFoundError("Not enough flower images in data/unsupervised/flowers/")


## Step 3 — Look at the images first

Before using an algorithm, inspect the data.

Think about variation in:

- colour
- flower shape
- brightness
- orientation
- background
- image composition


In [ ]:
rng = np.random.default_rng(42)
n_examples = min(12, len(image_files))
example_indices = rng.choice(len(image_files), size=n_examples, replace=False)

fig, axes = plt.subplots(3, 4, figsize=(12, 9))
axes = np.atleast_1d(axes).ravel()

for ax in axes:
    ax.axis("off")

for ax, index in zip(axes, example_indices):
    path = image_files[index]
    image = Image.open(path).convert("RGB")
    ax.imshow(image)
    ax.set_title(path.name, fontsize=9)
    ax.axis("off")

plt.suptitle("Example flower images", fontsize=16)
plt.tight_layout()
plt.show()


## Step 4 — Convert images into numerical features

K-means cannot directly understand the biological concept of a flower.

For this workshop we deliberately use **simple colour features**:

- RGB histograms
- mean RGB values
- RGB variation

This simplicity is useful for teaching because the clusters may reflect visual patterns rather than taxonomy.


In [ ]:
IMAGE_SIZE = (160, 160)
BINS = 16

def extract_colour_features(path):
    image = Image.open(path).convert("RGB")
    image = ImageOps.fit(image, IMAGE_SIZE)

    array = np.asarray(image, dtype=np.float32) / 255.0
    features = []

    for channel in range(3):
        histogram, _ = np.histogram(
            array[:, :, channel],
            bins=BINS,
            range=(0, 1),
            density=True,
        )
        features.extend(histogram)

    features.extend(array.mean(axis=(0, 1)))
    features.extend(array.std(axis=(0, 1)))

    return np.asarray(features, dtype=np.float32)


## Step 5 — Build the feature matrix

Each image becomes one row of numbers.


In [ ]:
X = np.vstack([
    extract_colour_features(path)
    for path in image_files
])

print("Feature matrix shape:", X.shape)
print("Images:", X.shape[0])
print("Features per image:", X.shape[1])


## Step 6 — Standardize the features

Standardization prevents features with larger numerical scales from dominating the analysis.


In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("Standardized feature matrix:", X_scaled.shape)


## Step 7 — Visualize feature space with PCA

We use **PCA only for visualization**.

> PCA is not the clustering algorithm in this activity.


In [ ]:
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

print(
    "Variance explained by PC1 + PC2:",
    f"{pca.explained_variance_ratio_.sum():.1%}"
)


In [ ]:
plt.figure(figsize=(8, 6))
plt.scatter(X_pca[:, 0], X_pca[:, 1], s=60)
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.title("Flower images in feature space")
plt.show()


### Pause

Do you already see possible groups?

Remember that this plot is only a two-dimensional projection of a higher-dimensional feature space.


## Step 8 — K-means clustering

K-means requires us to choose the number of clusters, **K**.

Start with:

```python
K = 2
```

This asks the algorithm to divide the images into two groups.


In [ ]:
K = 2

kmeans = KMeans(
    n_clusters=K,
    random_state=42,
    n_init=20,
)

cluster_labels = kmeans.fit_predict(X_scaled)

for cluster_id in range(K):
    n = np.sum(cluster_labels == cluster_id)
    print(f"Cluster {cluster_id}: {n} images")


## Step 9 — Show clusters in feature space


In [ ]:
plt.figure(figsize=(8, 6))
plt.scatter(
    X_pca[:, 0],
    X_pca[:, 1],
    c=cluster_labels,
    s=70,
)
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.title(f"K-means clustering (K = {K})")
plt.show()


## Step 10 — Inspect the original images in each cluster

The algorithm gives us cluster numbers. It does **not** tell us what those clusters mean biologically.


In [ ]:
def show_clusters(files, labels, max_images_per_cluster=24):
    for cluster_id in sorted(np.unique(labels)):
        indices = np.where(labels == cluster_id)[0]
        members = [files[i] for i in indices][:max_images_per_cluster]

        columns = 4
        rows = math.ceil(len(members) / columns)

        fig, axes = plt.subplots(
            rows,
            columns,
            figsize=(12, 3 * rows),
            squeeze=False,
        )
        axes = axes.ravel()

        for ax in axes:
            ax.axis("off")

        for ax, path in zip(axes, members):
            image = Image.open(path).convert("RGB")
            ax.imshow(image)
            ax.set_title(path.name, fontsize=8)
            ax.axis("off")

        fig.suptitle(f"Cluster {cluster_id}", fontsize=16)
        plt.tight_layout()
        plt.show()

show_clusters(image_files, cluster_labels)


## Discuss with your group

What seems to separate the clusters?

- flower colour?
- brightness?
- background?
- flower shape?
- image composition?
- several things together?

### Question

> Are these clusters automatically biological species?

Not necessarily.


# Experiment — Change K

Try changing K to 3 and then 4.

For each value:

1. Run K-means again.
2. Look at the PCA plot.
3. Inspect the images.
4. Interpret the clusters.


In [ ]:
K = 3  # Try 3, then 4

kmeans = KMeans(
    n_clusters=K,
    random_state=42,
    n_init=20,
)

cluster_labels = kmeans.fit_predict(X_scaled)

plt.figure(figsize=(8, 6))
plt.scatter(X_pca[:, 0], X_pca[:, 1], c=cluster_labels, s=70)
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.title(f"K-means clustering (K = {K})")
plt.show()

show_clusters(image_files, cluster_labels)


# Step 11 — How many clusters?

One simple diagnostic is the **elbow method**.

Inertia measures how tightly observations are grouped around their cluster centres.

Lower inertia means tighter clusters, but inertia always decreases as K increases.

We therefore look for a point where adding more clusters gives progressively smaller improvements.


In [ ]:
MAX_K = min(10, len(image_files) - 1)
K_VALUES = range(1, MAX_K + 1)
inertia_values = []

for k in K_VALUES:
    model = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=20,
    )
    model.fit(X_scaled)
    inertia_values.append(model.inertia_)


In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(list(K_VALUES), inertia_values, marker="o")
plt.xlabel("Number of clusters (K)")
plt.ylabel("Inertia")
plt.title("Elbow plot")
plt.xticks(list(K_VALUES))
plt.show()


## Think before continuing

Based only on the images, PCA plot, clustering results, and elbow plot:

> How many groups do you think are present?


# Final reveal

The dataset was actually sampled from **six Campanula species**.

The clustering algorithm was never given that information.

Now try:

```python
K = 6
```

### Question

> Does K-means with K = 6 recover six biological species?


In [ ]:
K = 6

kmeans = KMeans(
    n_clusters=K,
    random_state=42,
    n_init=20,
)

cluster_labels = kmeans.fit_predict(X_scaled)

plt.figure(figsize=(8, 6))
plt.scatter(X_pca[:, 0], X_pca[:, 1], c=cluster_labels, s=70)
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.title(f"K-means clustering (K = {K})")
plt.show()

show_clusters(image_files, cluster_labels)


# Discussion

Even if the dataset contains six biological species, K-means does not know the concept of a species.

It only sees the numerical features we provided. Here, those features are mainly based on colour.

Therefore, clusters may reflect colour, brightness, background, or composition rather than taxonomy.

## Compare with supervised learning

**Supervised learning:** We define the biological classes first.

**Unsupervised learning:** The algorithm finds patterns, but humans must interpret what those patterns mean.

# Take-home message

> **A cluster is a pattern in feature space. It is not automatically a biological group.**

> **AI outputs are evidence to evaluate, not biological truth to accept.**
